In [2]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from xgboost import XGBClassifier


In [3]:
df = pd.read_csv('../data/raw/creditcard.csv')

# sort by time
df = df.sort_values("Time")

# split manually
train_size = int(len(df) * 0.7)

train_df = df.iloc[:train_size]
test_df = df.iloc[train_size:]

#preprocessing
# separate features and target
X_train = train_df.drop(columns=["Class"])
y_train = train_df["Class"]

X_test = test_df.drop(columns=["Class"])
y_test = test_df["Class"]

X_train = X_train.drop(columns=["Time"])
X_test = X_test.drop(columns=["Time"])

scaler = StandardScaler()
X_train['Amount'] = scaler.fit_transform(X_train[['Amount']])
X_test['Amount'] = scaler.transform(X_test[['Amount']])

model = LogisticRegression(class_weight="balanced", max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)


xgb_model = XGBClassifier(
    scale_pos_weight= (198980 / 384),  # IMPORTANT FIX
    n_estimators=300,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    use_label_encoder=False,
    eval_metric="logloss"
)

xgb_model.fit(X_train, y_train)
y_prob = xgb_model.predict_proba(X_test)[:, 1]

y_pred_custom = (y_prob > 0.1).astype(int)



# print(confusion_matrix(y_test, y_pred_custom))
# print(classification_report(y_test, y_pred_custom))



E:\Data_Drift_Detector\venv\lib\site-packages\xgboost\training.py:200: UserWarning: [19:07:18] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [4]:
reference_data = train_df.copy()
current_data = test_df.copy()

reference_data = reference_data.drop(columns=["Class"])
current_data = current_data.drop(columns=["Class"])


In [1]:
from evidently.report import Report
from evidently.metric_preset import DataDriftPreset

In [5]:
from evidently.report import Report
from evidently.metric_preset import DataDriftPreset

report = Report(metrics=[DataDriftPreset()])

report.run(
    reference_data=reference_data,
    current_data=current_data
)

report.save_html("drift_report.html")

In [6]:
# Create Artificial Drift
current_data_drifted = current_data.copy()

# Increase transaction amount artificially
current_data_drifted["Amount"] = current_data_drifted["Amount"] * 3

In [7]:
# Run drift detection again

report_drifted = Report(metrics=[DataDriftPreset()])

report_drifted.run(
    reference_data=reference_data,
    current_data=current_data_drifted
)

report_drifted.save_html("drift_report_drifted.html")

In [8]:
current_data_heavy_drift = current_data.copy()

# Strong drift in multiple features
current_data_heavy_drift["Amount"] = current_data_heavy_drift["Amount"] * 5
current_data_heavy_drift["V1"] = current_data_heavy_drift["V1"] * 2
current_data_heavy_drift["V2"] = current_data_heavy_drift["V2"] * -1

In [9]:
report_heavy = Report(metrics=[DataDriftPreset()])

report_heavy.run(
    reference_data=reference_data,
    current_data=current_data_heavy_drift
)

report_heavy.save_html("drift_report_heavy.html")